In [22]:
import csv
from pathlib import Path

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_deepseek import ChatDeepSeek

In [ ]:
# ---------------------------------------------------------------------------
#  FILE LOCATIONS
# ---------------------------------------------------------------------------

BASE_DIR = Path(r"C:\Users\Phase2_TKH\dummy_data_for_week9_homework").resolve().parent

TRANSACTION_FILE = BASE_DIR / "orders.csv"
RETURN_POLICY_FILE = BASE_DIR / "return_policy.txt"
LOG_FILE = BASE_DIR / "tool_log.txt"


In [ ]:
# ---------------------------------------------------------------------------
#  LOGGING
# ---------------------------------------------------------------------------

def log_tool_call(tool_name, arguments, result):
    """Log every tool call, arguments, and result."""

    with open(LOG_FILE, "a", encoding="utf-8") as file:
        file.write("\n" + "=" * 50 + "\n")
        file.write(f"Tool: {tool_name}\n")
        file.write(f"Arguments: {arguments}\n")
        file.write(f"Result: {result}\n")


In [ ]:
# ---------------------------------------------------------------------------
#  TOOL 1 - LOOK UP AN ORDER
# ---------------------------------------------------------------------------

@tool
def lookup_order(order_id: str) -> str:
    """Look up an order using its order ID, such as ORD-1001."""

    with open(
        ORDER_FILE,
        "r",
        encoding="utf-8-sig"
    ) as file:

        reader = csv.DictReader(file)

        for row in reader:

            if row["order_id"].strip().lower() == order_id.strip().lower():

                result = (
                    f"Order ID: {row['order_id']}\n"
                    f"Customer: {row['customer_name']}\n"
                    f"Product: {row['product']}\n"
                    f"Order Date: {row['order_date']}\n"
                    f"Status: {row['status']}\n"
                    f"Tracking Number: {row['tracking_number']}\n"
                    f"Total Amount: ${row['total_amount']}"
                )

                log_tool_call(
                    "lookup_order",
                    {"order_id": order_id},
                    result
                )

                return result

    result = f"No order found for {order_id}."

    log_tool_call(
        "lookup_order",
        {"order_id": order_id},
        result
    )

    return result





In [ ]:
# ---------------------------------------------------------------------------
# TOOL 2 - CHECK RETURN POLICY
#
# This tool reads the actual return_policy.txt file.
# ---------------------------------------------------------------------------

@tool
def check_return_policy(question: str) -> str:
    """Answer a customer question using the return_policy.txt file."""

    with open(
        RETURN_POLICY_FILE,
        "r",
        encoding="utf-8"
    ) as file:

        policy = file.read()

    result = (
        f"Customer question: {question}\n\n"
        f"Return and Shipping Policy:\n{policy}"
    )

    log_tool_call(
        "check_return_policy",
        {"question": question},
        result
    )

    return result

In [ ]:
# ---------------------------------------------------------------------------
#  TOOL 3 - HIGH-RISK REFUND REQUEST
# ---------------------------------------------------------------------------

@tool
def request_refund(order_id: str, reason: str) -> str:
    """Request a refund. Human approval is required before this action."""

    print("\n" + "=" * 50)
    print("HUMAN APPROVAL REQUIRED")
    print("=" * 50)

    print(f"Order ID: {order_id}")
    print(f"Reason: {reason}")

    approval = input(
        "Approve this refund? Type 'yes' to approve: "
    )

    if approval.lower().strip() == "yes":

        result = (
            f"Refund request for {order_id} "
            f"was approved by a human. "
            f"This is a simulated refund."
        )

    else:

        result = (
            f"Refund request for {order_id} "
            f"was NOT approved."
        )

    log_tool_call(
        "request_refund",
        {
            "order_id": order_id,
            "reason": reason
        },
        result
    )

    return result

In [ ]:
# ---------------------------------------------------------------------------
#  SET UP THE MODEL
# ---------------------------------------------------------------------------

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    temperature=0,
)





In [ ]:
# ---------------------------------------------------------------------------
#  BUILD THE AGENT
# ---------------------------------------------------------------------------

agent = create_agent(
    model,
    tools=[
        lookup_order,
        check_return_policy,
        request_refund
    ],
    system_prompt="""
You are a helpful customer support assistant.

You can help customers with:

- Looking up orders
- Checking order status
- Tracking orders
- Return and shipping policy questions
- Refund requests

Rules:

1. Use lookup_order when the customer asks about
   an order ID such as ORD-1001.

2. Use lookup_order when the customer asks about:
   - Order status
   - Product
   - Order date
   - Tracking number
   - Total amount
   - Customer information

3. Use check_return_policy when the customer asks about:
   - Returns
   - Refund policies
   - Cancellations
   - Shipping
   - Shipping delays
   - Damaged items
   - Missing items
   - Warranty

4. Use request_refund when the customer wants a refund.

5. A refund is a high-risk action.
   The request_refund tool requires human approval.
   Never skip the approval step.

6. Do not invent information that is not in the files.

7. Be concise and helpful.
"""
)

In [ ]:
# ---------------------------------------------------------------------------
#  PRINT THE AGENT TRACE
# ---------------------------------------------------------------------------

def print_trace(messages):

    for message in messages:

        label = type(message).__name__

        if getattr(message, "tool_calls", None):

            for tc in message.tool_calls:

                print(
                    f"[{label}] requested tool call: "
                    f"{tc['name']}({tc['args']})"
                )

        elif label == "ToolMessage":

            print(
                f"[{label}] result: {message.content}"
            )

        else:

            print(
                f"[{label}] {message.content}"
            )



In [ ]:
# ---------------------------------------------------------------------------
#  RUN IT
# ---------------------------------------------------------------------------

def main():

    print("Customer Support Triage Agent")
    print("--------------------------------")

    user_input = input(
        "Customer> "
    )

    result = agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": user_input
            }
        ]
    })

    print(
        f"\nModel> {result['messages'][-1].content}"
    )

    print("\n--- Full trace ---")

    print_trace(
        result["messages"]
    )


In [ ]:
# ---------------------------------------------------------------------------
#  START PROGRAM
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    main()

Customer Support Triage Agent
--------------------------------

Model> Hi there! 👋 I'm here to help with anything related to your orders — whether it's checking an order status, tracking a shipment, or questions about returns, refunds, and shipping policies.

Just let me know what you need! If you have an order ID handy (like ORD-1001), that'll help me look things up quickly.

--- Full trace ---
[HumanMessage] 
[AIMessage] Hi there! 👋 I'm here to help with anything related to your orders — whether it's checking an order status, tracking a shipment, or questions about returns, refunds, and shipping policies.

Just let me know what you need! If you have an order ID handy (like ORD-1001), that'll help me look things up quickly.
